In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("retail_cleaned.csv",low_memory=False)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [7]:
dim_time = pd.DataFrame()
dim_time['InvoiceDate'] = df['InvoiceDate']
dim_time['Year'] = dim_time['InvoiceDate'].dt.year
dim_time['Month'] = dim_time['InvoiceDate'].dt.month
dim_time['MonthName'] = dim_time['InvoiceDate'].dt.month_name()
dim_time['Day'] = dim_time['InvoiceDate'].dt.day

dim_time = dim_time.drop_duplicates().reset_index(drop=True)
dim_time.head()


,InvoiceDate,Year,Month,MonthName,Day
0,2010-12-01 08:26:00,2010,12,December,1
1,2010-12-01 08:28:00,2010,12,December,1
2,2010-12-01 08:34:00,2010,12,December,1
3,2010-12-01 08:35:00,2010,12,December,1
4,2010-12-01 08:45:00,2010,12,December,1


In [9]:
dim_customer=(
    df[['CustomerID','Country']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_customer.head()

,CustomerID,Country
0,17850.0,United Kingdom
1,13047.0,United Kingdom
2,12583.0,France
3,13748.0,United Kingdom
4,15100.0,United Kingdom


In [11]:
dim_product = (
    df[['StockCode', 'Description', 'UnitPrice']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product.head()


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [13]:
fact_sales = df[[
    'InvoiceNo',
    'CustomerID',
    'StockCode',
    'Quantity',
    'UnitPrice',
    'TotalAmount',
    'InvoiceDate'
]]
fact_sales.head()


,InvoiceNo,CustomerID,StockCode,Quantity,UnitPrice,TotalAmount,InvoiceDate
0,536365,17850.0,85123A,6,2.55,15.30,2010-12-01 08:26:00
1,536365,17850.0,71053,6,3.39,20.34,2010-12-01 08:26:00
2,536365,17850.0,84406B,8,2.75,22.00,2010-12-01 08:26:00
3,536365,17850.0,84029G,6,3.39,20.34,2010-12-01 08:26:00
4,536365,17850.0,84029E,6,3.39,20.34,2010-12-01 08:26:00


In [26]:
monthly_sales = (
    fact_sales
    .groupby(pd.Grouper(key='InvoiceDate', freq='ME'))['TotalAmount']
    .sum()
    .reset_index()
)

monthly_sales.head()

,InvoiceDate,TotalAmount
0,2010-12-31,821452.730
1,2011-01-31,689811.610
2,2011-02-28,522545.560
3,2011-03-31,716215.260
4,2011-04-30,536968.491


In [28]:
sales_by_country=(
    fact_sales
    .merge(dim_customer, on='CustomerID')
    .groupby('Country')['TotalAmount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sales_by_country.head()

,Country,TotalAmount
0,United Kingdom,9039926.554
1,EIRE,2020164.370
2,France,1963836.220
3,Switzerland,1812124.760
4,Portugal,1788277.750
